In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import os
import keyboard, time

In [2]:
env_name = "Pendulum-v1"  # continuous action space env
env = gym.make(env_name)

1) Train PPO expert policy

In [4]:
print("Training PPO expert...")
model = PPO(
    "MlpPolicy", 
    env_name, 
    # verbose=1, 
    # learning_rate=3e-4,
    # n_steps=2048,
    # batch_size=64,
    # n_epochs=10,
    # gamma=0.99,
    # gae_lambda=0.95,
    # clip_range=0.2,
    # ent_coef=0.0,
    device="cuda" if torch.cuda.is_available() else "auto"
)

# Train the model
model.learn(total_timesteps=500000)
model.save("ppo_pendulum_expert")
print("PPO expert trained and saved!")

Training PPO expert...
PPO expert trained and saved!


1.1.Evaluate the expert

In [ ]:
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Expert mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

Expert mean reward: -306.25 +/- 305.37


1.2 Render the environment

In [10]:
def render_policy_interactive(policy, env_name, policy_type="bc", max_steps=1000, 
                              render_mode="human", sleep_time=0.02, 
                              device="cuda" if torch.cuda.is_available() else "cpu"):
    """
    Render and interact with a policy in real-time.
    
    Args:
        policy: The policy model (either SB3 model or PyTorch nn.Module)
        env_name: Name of the gym environment
        policy_type: "sb3" for Stable-Baselines3 model, "bc" for PyTorch BC model
        max_steps: Maximum steps per episode
        render_mode: "human" for window, "rgb_array" for array output
        sleep_time: Time between steps for visualization
        device: Device for PyTorch models
    """
    env = gym.make(env_name, render_mode=render_mode)
    obs, _ = env.reset()
    
    print(f"\n--- Interactive Rendering ({policy_type.upper()} policy) ---")
    print("Press SPACE to step, ESC to exit, R to reset")
    print("Press A for automatic continuous mode, S to stop automatic mode")
    
    step_count = 0
    total_reward = 0
    automatic_mode = False
    
    while step_count < max_steps:
        # Clear screen and show status
        print("\033[2J\033[H", end="")  # Clear console
        print(f"Step: {step_count}/{max_steps}")
        print(f"Total Reward: {total_reward:.2f}")
        print(f"Mode: {'AUTOMATIC' if automatic_mode else 'MANUAL'}")
        print("-" * 50)
        
        # Get action from policy
        if policy_type == "sb3":
            # Stable-Baselines3 model
            action, _ = policy.predict(obs, deterministic=True)
        elif policy_type == "bc":
            # PyTorch BC model
            with torch.no_grad():
                obs_tensor = torch.from_numpy(obs.astype(np.float32)).to(device).unsqueeze(0)
                action = policy(obs_tensor).cpu().numpy()[0]
        elif policy_type == "random":
            # Random policy
            action = env.action_space.sample()
        else:
            raise ValueError(f"Unknown policy_type: {policy_type}")
        
        # Take step in environment
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        step_count += 1
        
        # # Check for user input (non-blocking)
        # if keyboard.is_pressed('space'):
        #     automatic_mode = False
        #     time.sleep(0.3)  # Debounce
        # elif keyboard.is_pressed('a'):
        #     automatic_mode = True
        #     time.sleep(0.3)
        # elif keyboard.is_pressed('s'):
        #     automatic_mode = False
        #     time.sleep(0.3)
        # elif keyboard.is_pressed('r'):
        #     obs, _ = env.reset()
        #     total_reward = 0
        #     step_count = 0
        #     print("Environment reset!")
        #     time.sleep(0.3)
        # elif keyboard.is_pressed('esc'):
        #     print("Exiting...")
        #     break
        
        # # Control playback speed
        # if not automatic_mode:
        #     print("\nPress SPACE for next step...")
        #     while not keyboard.is_pressed('space') and not keyboard.is_pressed('esc') and not keyboard.is_pressed('a'):
        #         time.sleep(0.1)
        #     time.sleep(0.2)  # Debounce
        # else:
        #     time.sleep(sleep_time)
        
        # Check if episode ended
        if terminated or truncated:
            print(f"\nEpisode finished! Total Reward: {total_reward:.2f}")
            obs, _ = env.reset()
            total_reward = 0
            step_count = 0
            time.sleep(1)
    
    env.close()
    print("Rendering finished!")

In [ ]:
render_policy_interactive(model, env_name=env_name, policy_type="sb3")


--- Interactive Rendering (SB3 policy) ---
Press SPACE to step, ESC to exit, R to reset
Press A for automatic continuous mode, S to stop automatic mode
Step: 0/1000
Total Reward: 0.00
Mode: MANUAL
--------------------------------------------------
Step: 1/1000
Total Reward: -3.68
Mode: MANUAL
--------------------------------------------------
Step: 2/1000
Total Reward: -7.80
Mode: MANUAL
--------------------------------------------------
Step: 3/1000
Total Reward: -12.71
Mode: MANUAL
--------------------------------------------------
Step: 4/1000
Total Reward: -18.86
Mode: MANUAL
--------------------------------------------------
Step: 5/1000
Total Reward: -26.65
Mode: MANUAL
--------------------------------------------------
Step: 6/1000
Total Reward: -36.41
Mode: MANUAL
--------------------------------------------------
Step: 7/1000
Total Reward: -48.40
Mode: MANUAL
--------------------------------------------------
Step: 8/1000
Total Reward: -60.96
Mode: MANUAL
--------------------

KeyboardInterrupt: 

2) Collect dataset from expert policy

In [5]:
print("Collecting dataset from expert...")
states, actions = [], []
env = gym.make(env_name)

# Collect trajectories
n_trajectories = 400
max_steps_per_traj = 200
max_total_samples = 20000

for episode in range(n_trajectories):
    obs, _ = env.reset()
    done = False
    episode_states = []
    episode_actions = []
    step_count = 0
    
    while not done and step_count < max_steps_per_traj and len(states) < max_total_samples:
        # Get action from expert policy
        action, _ = model.predict(obs, deterministic=True)
        
        # Store state and action
        episode_states.append(obs.astype(np.float32))
        episode_actions.append(action.astype(np.float32))
        
        # Step environment
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        step_count += 1
    
    # Add episode data to overall dataset
    states.extend(episode_states)
    actions.extend(episode_actions)
    
    if len(states) >= max_total_samples:
        break

states = np.array(states[:max_total_samples])
actions = np.array(actions[:max_total_samples])

print(f"Collected dataset: {len(states)} samples")

Collected dataset: 20000 samples


3. Create dataloader

In [6]:
ds = TensorDataset(torch.from_numpy(states), torch.from_numpy(actions))
loader = DataLoader(ds, batch_size=256, shuffle=True)

4. Define BC Model

In [7]:
class Policy(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 256), 
            nn.ReLU(),
            nn.Linear(256, 256), 
            nn.ReLU(),
            nn.Linear(256, act_dim)
        )
    
    def forward(self, x): 
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bc = Policy(env.observation_space.shape[0], env.action_space.shape[0]).to(device)
opt = torch.optim.Adam(model_bc.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()

5. Train BC Model

In [8]:
print("Training behavioral cloning model...")
for epoch in range(12):
    total_loss = 0
    num_batches = 0
    model_bc.train()
    
    for s_batch, a_batch in loader:
        s_batch = s_batch.to(device)
        a_batch = a_batch.to(device)
        
        pred = model_bc(s_batch)
        loss = loss_fn(pred, a_batch)
        
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch}: loss = {avg_loss:.6f}")

# Save BC model
torch.save(model_bc.state_dict(), "bc_pendulum_model.pth")
print("BC model saved!")

Training behavioral cloning model...
Epoch 0: loss = 0.183762
Epoch 1: loss = 0.050424
Epoch 2: loss = 0.024421
Epoch 3: loss = 0.018038
Epoch 4: loss = 0.014331
Epoch 5: loss = 0.011441
Epoch 6: loss = 0.009675
Epoch 7: loss = 0.007793
Epoch 8: loss = 0.006586
Epoch 9: loss = 0.005550
Epoch 10: loss = 0.004679
Epoch 11: loss = 0.004220
BC model saved!


In [11]:
render_policy_interactive(model_bc, env_name=env_name, policy_type="bc")

/home/prakash/miniforge3/envs/flow_matching/lib/python3.9/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists



--- Interactive Rendering (BC policy) ---
Press SPACE to step, ESC to exit, R to reset
Press A for automatic continuous mode, S to stop automatic mode
Step: 0/1000
Total Reward: 0.00
Mode: MANUAL
--------------------------------------------------
Step: 1/1000
Total Reward: -2.76
Mode: MANUAL
--------------------------------------------------
Step: 2/1000
Total Reward: -5.52
Mode: MANUAL
--------------------------------------------------
Step: 3/1000
Total Reward: -8.41
Mode: MANUAL
--------------------------------------------------
Step: 4/1000
Total Reward: -11.58
Mode: MANUAL
--------------------------------------------------
Step: 5/1000
Total Reward: -15.25
Mode: MANUAL
--------------------------------------------------
Step: 6/1000
Total Reward: -19.72
Mode: MANUAL
--------------------------------------------------
Step: 7/1000
Total Reward: -25.41
Mode: MANUAL
--------------------------------------------------
Step: 8/1000
Total Reward: -32.68
Mode: MANUAL
----------------------

KeyboardInterrupt: 

# Flow policy BC training

In [12]:
import time
import torch

from torch import nn, Tensor

# flow_matching
from flow_matching.path.scheduler import CondOTScheduler
from flow_matching.path import AffineProbPath
from flow_matching.solver import Solver, ODESolver
from flow_matching.utils import ModelWrapper

# visualization
import matplotlib.pyplot as plt

from matplotlib import cm


# To avoide meshgrid warning
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='torch')

In [13]:
if torch.cuda.is_available():
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

Using cpu.


In [14]:
torch.manual_seed(42)

Model

In [15]:
# Activation class
class Swish(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x: Tensor) -> Tensor: 
        return torch.sigmoid(x) * x

# Model class
class MLP(nn.Module):
    def __init__(self, input_dim: int = 2, state_dim: int = 0, time_dim: int = 1, hidden_dim: int = 128):
        super().__init__()
        
        self.input_dim = input_dim
        self.time_dim = time_dim
        self.hidden_dim = hidden_dim
        self.state_dim = state_dim

        
        self.main = nn.Sequential(
            nn.Linear(input_dim + time_dim + state_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, hidden_dim),
            Swish(),
            nn.Linear(hidden_dim, input_dim),
        )
    

    def forward(self, x: Tensor, t: Tensor, s: Tensor) -> Tensor:
        sz = x.size()
        x = x.reshape(-1, self.input_dim)
        t = t.reshape(-1, self.time_dim).float()

        t = t.reshape(-1, 1).expand(x.shape[0], 1)
        # handle state/context if provided (and if state_dim > 0)
        if self.state_dim > 0:
            if s is None:
                raise ValueError("state_dim > 0 but no state `s` provided to forward().")
            s = s.reshape(-1, self.state_dim).float()         # [B_state, state_dim]
            if s.shape[0] != x.shape[0]:
                # allow broadcasting when s has fewer batch entries (e.g. one per original sample)
                if x.shape[0] % s.shape[0] != 0:
                    raise ValueError(f"Cannot expand state batch {s.shape[0]} to match x batch {x.shape[0]}.")
                k = x.shape[0] // s.shape[0]
                s = s.unsqueeze(1).repeat(1, k, 1).reshape(-1, self.state_dim)  # [B_flat, state_dim]
            # now s.shape[0] == x.shape[0]

            h = torch.cat([x, t, s], dim=1)
        else:
            h = torch.cat([x, t], dim=1)
            
        output = self.main(h)
        
        return output.reshape(*sz)

In [16]:
loader_iter = iter(loader) 

## Train Velocity Flow Matching model

In [17]:
# training arguments
lr = 0.001
batch_size = 4096
iterations = 20001
print_every = 2000 
hidden_dim = 512

input_dim = env.action_space.shape[0]
state_dim = env.observation_space.shape[0]

# velocity field model init
vf = MLP(input_dim=input_dim, state_dim=state_dim, time_dim=1, hidden_dim=hidden_dim).to(device) 

# instantiate an affine path object
path = AffineProbPath(scheduler=CondOTScheduler())

# init optimizer
optim = torch.optim.Adam(vf.parameters(), lr=lr) 

# train
start_time = time.time()
for i in range(iterations):
    optim.zero_grad() 

    # sample data (user's responsibility): in this case, (X_0,X_1) ~ pi(X_0,X_1) = N(X_0|0,I)q(X_1)
    try:
        s_batch, x_1 = next(loader_iter)   # get a batch of (states, expert actions) -> x_1
    except StopIteration:
        loader_iter = iter(loader)
        s_batch, x_1 = next(loader_iter)
    x_0 = torch.randn_like(x_1).to(device)

    # sample time (user's responsibility)
    t = torch.rand(x_1.shape[0]).to(device) 

    # sample probability path
    path_sample = path.sample(t=t, x_0=x_0, x_1=x_1)

    # flow matching l2 loss
    loss = torch.pow( vf(path_sample.x_t,path_sample.t,s_batch) - path_sample.dx_t, 2).mean() 

    # optimizer step
    loss.backward() # backward
    optim.step() # update
    
    # log loss
    if (i+1) % print_every == 0:
        elapsed = time.time() - start_time
        print('| iter {:6d} | {:5.2f} ms/step | loss {:8.3f} ' 
              .format(i+1, elapsed*1000/print_every, loss.item())) 
        start_time = time.time()

| iter   2000 |  5.45 ms/step | loss    0.128 
| iter   4000 |  5.39 ms/step | loss    0.045 
| iter   6000 |  5.54 ms/step | loss    0.056 
| iter   8000 |  5.48 ms/step | loss    0.041 
| iter  10000 |  5.35 ms/step | loss    0.069 
| iter  12000 |  5.33 ms/step | loss    0.026 
| iter  14000 |  5.38 ms/step | loss    0.082 
| iter  16000 |  5.41 ms/step | loss    0.021 
| iter  18000 |  5.39 ms/step | loss    0.083 
| iter  20000 |  5.38 ms/step | loss    0.016 


#### Sample from trained model

In [19]:
class WrappedModel(ModelWrapper):
    def forward(self, x: torch.Tensor, t: torch.Tensor, s: torch.Tensor, **extras):
        return self.model(x, t, s, **extras)

wrapped_vf = WrappedModel(vf)

In [ ]:
# step size for ode solver
step_size = 0.05

norm = cm.colors.Normalize(vmax=50, vmin=0)

batch_size = 50000  # batch size
eps_time = 1e-2
T = torch.linspace(0,1,10)  # sample times
T = T.to(device=device)

x_init = torch.randn((batch_size, 2), dtype=torch.float32, device=device)
solver = ODESolver(velocity_model=wrapped_vf)  # create an ODESolver class
sol = solver.sample(time_grid=T, x_init=x_init, method='midpoint', step_size=step_size, return_intermediates=True)  # sample from the model

In [ ]:

# ---------- Evaluate flow policy in environment (returns mean ep return) ----------
def evaluate_flow_policy(env_name: str,
                         vf,                 # callable or nn.Module: vf(x,t,s) -> v
                         ODESolver: ODESolver,          # class/constructor for your solver
                         device: str = None,
                         episodes: int = 10,
                         K: int = 8,         # samples per state
                         T: torch.Tensor = None,
                         step_size: float = 0.05,
                         solver_method: str = "midpoint",
                         max_steps: int = 1000,
                         render: bool = False):
    """
    Evaluate a flow policy by sampling actions via ODE solver integration.

    Args:
        env_name: gymnasium env id (e.g. "Pendulum-v1")
        vf: velocity field function/module with signature vf(x, t, s)
        ODESolver: class or factory that accepts velocity_model=callable
        device: "cuda"/"cpu" or None to choose automatically
        episodes: number of episodes to evaluate
        K: number of Monte-Carlo action samples per observation
        T: time grid tensor (1D) used by solver; if None uses torch.linspace(0,1,10)
        step_size: solver internal step size
        solver_method: e.g. "midpoint"
        max_steps: max steps per episode
        render: whether to call env.render() (depends on env and render_mode)
    Returns:
        (mean_return, std_return)
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    if T is None:
        T = torch.linspace(0.0, 1.0, 10, device=device)

    env = gym.make(env_name, render_mode="human" if render else None)
    action_dim = env.action_space.shape[0]
    state_dim = env.observation_space.shape[0]

    returns = []
    for ep in range(episodes):
        obs, _ = env.reset()
        done = False
        ep_ret = 0.0
        step = 0

        while not done and step < max_steps:
            # prepare state tensor
            s = torch.from_numpy(np.asarray(obs, dtype=np.float32)).to(device).unsqueeze(0)  # [1, state_dim]
            # repeat state K times
            s_rep = s.repeat(K, 1)  # [K, state_dim]

            # sample noise initial points x_init ~ N(0,I)
            x_init = torch.randn((K, action_dim), dtype=torch.float32, device=device)

            # instantiate solver with this velocity
            solver = ODESolver(velocity_model=wrapped_vf)

            # integrate and extract final samples
    
            sol = solver.sample(time_grid=T, x_init=x_init, s = s_rep, method=solver_method,
                                step_size=step_size, return_intermediates=False)

            final = sol   # [K, action_dim]

            # aggregate K samples into one action (mean here)
            action_tensor = final.view(K, action_dim).mean(dim=0)  # [action_dim]
            action = action_tensor.cpu().numpy()


            # step env (gymnasium API)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = bool(terminated or truncated)
            ep_ret += float(reward)

            obs = next_obs
            step += 1

            if render:
                # some envs require render() call or return frame from step depending on render_mode
                try:
                    env.render()
                except Exception:
                    pass

        returns.append(ep_ret)
        print(f"Episode {ep+1}/{episodes} return: {ep_ret:.3f}")

    env.close()
    returns = np.array(returns, dtype=np.float32)
    return float(returns.mean()), float(returns.std())



In [46]:
# Example: evaluate 5 episodes
mean_r, std_r = evaluate_flow_policy(
    env_name=env_name,
    vf=wrapped_vf,         # callable/module: vf(x,t,s)
    ODESolver=ODESolver,       # your ODESolver class
    device=device,
    episodes=3,
    K=1,
    T=torch.linspace(0,1,2).to(device),
    # solver_method="euler",
    # step_size=1,
    render=True
)
print("Mean return:", mean_r, "std:", std_r)


Episode 1/3 return: -126.745
Episode 2/3 return: -117.959
Episode 3/3 return: -128.341
Mean return: -124.34871673583984 std: 4.564610481262207
